##### 1. Purpose

This notebook extends the Support Ticket NLP project from a linear classifier to a neural-network classifier.

The experiment keeps the text representation unchanged:

``` text

Ticket text
    ↓
Pretrained SentenceTransformer
all-MiniLM-L6-v2
    ↓
384-dimensional embedding
    ↓
Classifier

```

Notebook 08 used:

``` text

384 embeddings
      ↓
Logistic Regression
      ↓
4 ticket categories

```

This notebook evaluates:

``` text

384 embeddings
      ↓
Neural Network
384 → 64 → 32 → 4
      ↓
4 ticket categories

```

The controlled research question is:

Does a nonlinear neural-network classifier provide useful benefit over Logistic Regression when both models receive the same pretrained sentence embeddings?

##### 2. Concepts

This notebook reinforces:

- Sentence embeddings
- Multiclass classification
- Neural-network layers
- Linear transformations
- ReLU nonlinearity
- Logits
- CrossEntropyLoss
- Softmax
- Argmax
- Forward propagation
- Backpropagation
- Optimizer
- Epochs
- Mini-batches
- Validation
- Early stopping
- Overfitting

##### 3. Architecture

``` text

Support ticket
     ↓
Clean text
     ↓
all-MiniLM-L6-v2
     ↓
384 embedding features
     ↓
Linear(384, 64)
     ↓
ReLU
     ↓
Linear(64, 32)
     ↓
ReLU
     ↓
Linear(32, 4)
     ↓
4 logits
     ↓
Softmax
     ↓
Billing / Cancellation / Login / Technical

```

During training:

``` text

4 logits
   +
actual class ID
   ↓
CrossEntropyLoss
   ↓
Backpropagation
   ↓
Adam optimizer
   ↓
Updated weights and biases

```


##### 4. Install Dependencies

In [0]:
%pip install sentence-transformers

##### 5. Imports

In [0]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.preprocessing import (
    LabelEncoder,
)

from src.project_config import (
    RANDOM_SEED,
    CLEAN_TEXT_COL,
    TARGET_COL,
    TICKET_ID_COL,
)

from src.data_preparation import (
    load_modeling_dataset,
    validate_modeling_schema,
    split_modeling_dataset,
    to_pandas_modeling_data,
)

from src.feature_engineering import (
    load_embedding_model,
    build_embedding_features,
)

from src.model_training import (
    train_logistic_regression,
)

from src.model_evaluation import (
    calculate_classification_metrics,
    build_classification_report,
    build_confusion_matrix,
    build_prediction_results,
)

from src.neural_network import (
    TicketClassifierNN,
    set_torch_seed,
    create_tensor_dataset,
    create_data_loader,
    evaluate_loss_and_accuracy,
    train_neural_network,
    predict_neural_network,
)

##### 6. Reproducibility

In [0]:
set_torch_seed(
    RANDOM_SEED
)

In [0]:
print(
    "PyTorch version:",
    torch.__version__,
)

##### 7.  Load the Persisted Dataset

In [0]:
modeling_sdf = (
    load_modeling_dataset(
        spark
    )
)

validate_modeling_schema(
    modeling_sdf
)

In [0]:
(
    train_sdf,
    validation_sdf,
    test_sdf,
) = split_modeling_dataset(
    modeling_sdf
)

In [0]:
print(
    "Training rows:",
    train_sdf.count(),
)

print(
    "Validation rows:",
    validation_sdf.count(),
)

print(
    "Test rows:",
    test_sdf.count(),
)

##### 8. Convert Modeling Data to Pandas

In [0]:
train_pdf = (
    to_pandas_modeling_data(
        train_sdf
    )
)

validation_pdf = (
    to_pandas_modeling_data(
        validation_sdf
    )
)

test_pdf = (
    to_pandas_modeling_data(
        test_sdf
    )
)

In [0]:
X_train_text = (
    train_pdf[CLEAN_TEXT_COL]
    .fillna("")
    .astype(str)
)

X_validation_text = (
    validation_pdf[CLEAN_TEXT_COL]
    .fillna("")
    .astype(str)
)

X_test_text = (
    test_pdf[CLEAN_TEXT_COL]
    .fillna("")
    .astype(str)
)

In [0]:
#prepare labels

y_train = (
    train_pdf[TARGET_COL]
)

y_validation = (
    validation_pdf[TARGET_COL]
)

y_test = (
    test_pdf[TARGET_COL]
)

##### 9. Generate Sentence Embeddings

In [0]:
embedding_model = (
    load_embedding_model()
)

In [0]:
(
    X_train_embeddings,
    X_validation_embeddings,
    X_test_embeddings,
) = build_embedding_features(
    embedding_model,
    X_train_text,
    X_validation_text,
    X_test_text,
)

In [0]:
print(
    "Train embeddings:",
    X_train_embeddings.shape,
)

print(
    "Validation embeddings:",
    X_validation_embeddings.shape,
)

print(
    "Test embeddings:",
    X_test_embeddings.shape,
)

##### 10. Encode String Labels

In [0]:
#PyTorch's CrossEntropyLoss expects integer class indices but labels are strings.

label_encoder = LabelEncoder()

In [0]:
y_train_encoded = (
    label_encoder.fit_transform(
        y_train
    )
)

y_validation_encoded = (
    label_encoder.transform(
        y_validation
    )
)

y_test_encoded = (
    label_encoder.transform(
        y_test
    )
)

In [0]:
print(
    "Classes:",
    label_encoder.classes_,
)

In [0]:
for class_id, class_name in enumerate(
    label_encoder.classes_
):
    print(
        class_id,
        "→",
        class_name,
    )

##### 11. Create PyTorch Datasets

In [0]:
train_dataset = (
    create_tensor_dataset(
        X_train_embeddings,
        y_train_encoded,
    )
)

validation_dataset = (
    create_tensor_dataset(
        X_validation_embeddings,
        y_validation_encoded,
    )
)

test_dataset = (
    create_tensor_dataset(
        X_test_embeddings,
        y_test_encoded,
    )
)

In [0]:
X_train_tensor, y_train_tensor = (
    train_dataset.tensors
)

X_validation_tensor, y_validation_tensor = (
    validation_dataset.tensors
)

X_test_tensor, y_test_tensor = (
    test_dataset.tensors
)

In [0]:
print(
    "X_train_tensor:",
    X_train_tensor.shape,
)

print(
    "y_train_tensor:",
    y_train_tensor.shape,
)

print(
    "X_validation_tensor:",
    X_validation_tensor.shape,
)

print(
    "y_validation_tensor:",
    y_validation_tensor.shape,
)

print(
    "X_test_tensor:",
    X_test_tensor.shape,
)

print(
    "y_test_tensor:",
    y_test_tensor.shape,
)

Features
↓
float32

Labels
↓
long / integer class IDs

##### 12. Create DataLoaders

In [0]:
BATCH_SIZE = 16

In [0]:
train_loader = (
    create_data_loader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
)

validation_loader = (
    create_data_loader(
        dataset=validation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
)

test_loader = (
    create_data_loader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
)

Training is shuffled because gradient updates should see examples in varying mini-batch order.

Validation and test are not shuffled because we are only evaluating them.

train → shuffle=True

because we want training batches mixed between epochs.

But:

validation/test → shuffle=False

because we are not training on them.

##### 13.  Inspect One Batch

In [0]:
X_batch, y_batch = next(
    iter(train_loader)
)

In [0]:
print(
    "Input batch shape:",
    X_batch.shape,
)

print(
    "Target batch shape:",
    y_batch.shape,
)

##### 14. Configure the Neural Network

In [0]:
INPUT_SIZE = (
    X_train_embeddings.shape[1]
)

HIDDEN_SIZE_1 = 64
HIDDEN_SIZE_2 = 32

NUM_CLASSES = len(
    label_encoder.classes_
)

In [0]:
print(
    "Input size:",
    INPUT_SIZE,
)

print(
    "Number of classes:",
    NUM_CLASSES,
)

##### 15. Select Device

In [0]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device,
)

##### 16. Create the Neural Network

In [0]:
model = TicketClassifierNN(
    input_size=INPUT_SIZE,
    hidden_size_1=HIDDEN_SIZE_1,
    hidden_size_2=HIDDEN_SIZE_2,
    num_classes=NUM_CLASSES,
)

In [0]:
model = model.to(
    device
)

In [0]:
print(
    model
)

##### 17. Count Parameters

In [0]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

In [0]:
print(
    "Total parameters:",
    total_parameters,
)

print(
    "Trainable parameters:",
    trainable_parameters,
)

##### 18. Verify Forward Propagation

In [0]:
model.eval()

#Before training, always verify that a batch can pass through the network.

with torch.no_grad():

    initial_logits = model(
        X_batch.to(
            device
        )
    )

In [0]:
print(
    "Input batch shape:",
    X_batch.shape,
)

print(
    "Output logits shape:",
    initial_logits.shape,
)

##### 19 - Inspect One Ticket's Logits

In [0]:
print(
     initial_logits [0]
)

##### 20. Inspect Initial Probabilities

In [0]:
initial_probabilities = (
    torch.softmax(
        initial_logits,
        dim=1,
    )
)

In [0]:
print(
    initial_probabilities[0]
)

print(
    "Probability sum:",
    initial_probabilities[0].sum(),
)

##### 21. Configure Loss and Optimizer

In [0]:
criterion = nn.CrossEntropyLoss()

In [0]:
LEARNING_RATE = 0.001

In [0]:
optimizer = (
    torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
    )
)

##### 22. Inspect Initial Loss

In [0]:
initial_loss = criterion(
    initial_logits,
    y_batch.to(
        device
    ),
)

In [0]:
print(
    "Initial batch loss:",
    initial_loss.item(),
)

##### 23. Training Configuration

In [0]:
NUM_EPOCHS = 100
PATIENCE = 10

##### 24. Train the Neural Network

In [0]:
training_history = (
    train_neural_network(
        model=model,
        train_loader=train_loader,
        validation_loader=validation_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=NUM_EPOCHS,
        patience=PATIENCE,
    )
)

In [0]:
history_df = pd.DataFrame(
    training_history
)

In [0]:
display(
    history_df
)

##### 25. Inspect Final Training History

In [0]:
print(
    "Epochs completed:",
    len(history_df),
)

In [0]:
display(history_df.tail(10))

##### 26. Evaluate the Best Restored Model on Validation

In [0]:
(
    validation_loss,
    validation_accuracy,
) = evaluate_loss_and_accuracy(
    model=model,
    data_loader=validation_loader,
    criterion=criterion,
    device=device,
)

In [0]:
print(
    f"Validation Loss: "
    f"{validation_loss:.4f}"
)

print(
    f"Validation Accuracy: "
    f"{validation_accuracy:.4f}"
)

##### 27. Final Test Prediction

In [0]:
(
    test_prediction_ids,
    test_probabilities,
) = predict_neural_network(
    model=model,
    data_loader=test_loader,
    device=device,
)

In [0]:
test_predictions = (
    label_encoder.inverse_transform(
        test_prediction_ids
    )
)

##### 28. Calculate Test Metrics

In [0]:
nn_test_metrics = (
    calculate_classification_metrics(
        y_test,
        test_predictions,
    )
)

In [0]:
print(
    "Neural Network Test Metrics"
)

print(
    f"Accuracy: "
    f"{nn_test_metrics['accuracy']:.4f}"
)

print(
    f"Macro F1: "
    f"{nn_test_metrics['macro_f1']:.4f}"
)

print(
    f"Weighted F1: "
    f"{nn_test_metrics['weighted_f1']:.4f}"
)

##### 29. Classification Report

In [0]:
nn_classification_report_df = (
    build_classification_report(
        y_test,
        test_predictions,
    )
)

In [0]:
display(
    nn_classification_report_df
)

##### 30. Confusion Matrix

In [0]:
class_labels = list(
    label_encoder.classes_
)

In [0]:
nn_confusion_df = (
    build_confusion_matrix(
        y_test,
        test_predictions,
        class_labels,
    )
)

In [0]:
display(
    nn_confusion_df
)

##### 31. Build Prediction-Level Results

In [0]:
nn_test_results_df = (
    build_prediction_results(
        ticket_ids=test_pdf[
            TICKET_ID_COL
        ],
        text=test_pdf[
            CLEAN_TEXT_COL
        ],
        y_true=y_test,
        y_pred=test_predictions,
        probabilities=test_probabilities,
        class_labels=class_labels,
    )
)

In [0]:
display(
    nn_test_results_df
)

##### 32. Inspect Neural-Network Errors

In [0]:
nn_errors_df = (
    nn_test_results_df[
        ~nn_test_results_df[
            "is_correct"
        ]
    ]
    .copy()
)

In [0]:
print(
    "Neural Network Test Errors:",
    len(nn_errors_df),
)

In [0]:
display(
    nn_errors_df
)

##### 33. Controlled Comparison: Embeddings + Logistic Regression

In [0]:
embedding_lr_classifier = (
    train_logistic_regression(
        X_train_embeddings,
        y_train,
    )
)

In [0]:
embedding_lr_predictions = (
    embedding_lr_classifier.predict(
        X_test_embeddings
    )
)

In [0]:
embedding_lr_metrics = (
    calculate_classification_metrics(
        y_test,
        embedding_lr_predictions,
    )
)

In [0]:
embedding_lr_errors = int(
    np.sum(
        embedding_lr_predictions
        != np.asarray(y_test)
    )
)

In [0]:
nn_errors = int(
    np.sum(
        test_predictions
        != np.asarray(y_test)
    )
)

##### 34. Compare Linear vs Nonlinear Classifier

In [0]:
comparison_df = pd.DataFrame(
    [
        {
            "representation": (
                "Sentence Embeddings"
            ),
            "classifier": (
                "Logistic Regression"
            ),
            "relationship": (
                "Linear"
            ),
            "accuracy": (
                embedding_lr_metrics[
                    "accuracy"
                ]
            ),
            "macro_f1": (
                embedding_lr_metrics[
                    "macro_f1"
                ]
            ),
            "weighted_f1": (
                embedding_lr_metrics[
                    "weighted_f1"
                ]
            ),
            "test_errors": (
                embedding_lr_errors
            ),
        },
        {
            "representation": (
                "Sentence Embeddings"
            ),
            "classifier": (
                "Neural Network"
            ),
            "relationship": (
                "Nonlinear"
            ),
            "accuracy": (
                nn_test_metrics[
                    "accuracy"
                ]
            ),
            "macro_f1": (
                nn_test_metrics[
                    "macro_f1"
                ]
            ),
            "weighted_f1": (
                nn_test_metrics[
                    "weighted_f1"
                ]
            ),
            "test_errors": (
                nn_errors
            ),
        },
    ]
)

In [0]:
display(
    comparison_df
)

##### 35. Inspect Parameter Difference

In [0]:
print(
    "Neural Network Parameters:",
    trainable_parameters,
)

In [0]:
lr_parameter_count = (
    embedding_lr_classifier.coef_.size
    +
    embedding_lr_classifier.intercept_.size
)

In [0]:
print(
    "Logistic Regression "
    "Parameters:",
    lr_parameter_count,
)

##### 36. Key Learning — Linear vs Nonlinear

Logistic Regression learns approximately: $$ z_k=W_kx+b_k $$

for each category.

The neural network learns: $$ x \rightarrow W_1x+b_1 \rightarrow ReLU \rightarrow W_2h+b_2 \rightarrow ReLU \rightarrow W_3h+b_3 $$

The ReLU layers allow nonlinear combinations of the embedding features.

Therefore:

``` text

Embeddings + Logistic Regression
        ↓
semantic representation
+
linear classifier


Embeddings + Neural Network
        ↓
semantic representation
+
nonlinear classifier

```

##### 37. Key Learning — Binary vs Multiclass Neural Network

Telco neural network:

``` text

45 inputs
 ↓
hidden layers
 ↓
1 logit
 ↓
BCEWithLogitsLoss
 ↓
Churn / No Churn

```

Support Ticket NLP:

``` text

384 inputs
 ↓
hidden layers
 ↓
4 logits
 ↓
CrossEntropyLoss
 ↓
Billing
Cancellation
Login
Technical

```

So the neural-network training principles remain the same.

The major change is the classification task.

##### 38. Key Learning — Representation vs Classifier

This notebook reinforces an especially important architecture distinction:

``` text

TEXT
 ↓
REPRESENTATION
SentenceTransformer
 ↓
384 numerical features
 ↓
CLASSIFIER
Neural Network
 ↓
PREDICTION
Ticket category

```

The SentenceTransformer and the neural-network classifier are not the same model in this experiment.

MiniLM is being used as a pretrained feature generator.

PyTorch classifier is the model we trained using the 70 local training tickets.

##### 39. Key Learning — More Complex Is Not Automatically Better

Dataset contains only: 70 training tickets

yet the neural network contains: 26,852 trainable parameters

Embedding + Logistic Regression contains only about: 1,540 parameters

Therefore, if their validation/test performance is similar, the simpler model may be preferable because it is easier to:

- train
- debug
- explain
- deploy
- maintain

The purpose of Notebook 09 is therefore not to force the neural network to beat Logistic Regression.

It is to determine whether nonlinear classification adds value.

##### 40. Notebook Conclusion

- This notebook extended the Support Ticket NLP project from a linear embedding classifier to a neural-network classifier.
- The same pretrained all-MiniLM-L6-v2 sentence embeddings were used for both classifiers, allowing the experiment to isolate the effect of changing the learning algorithm.
- The neural network accepted 384-dimensional sentence embeddings and used the architecture:  384 → 64 → 32 → 4
- ReLU activation functions introduced nonlinear learning, while the final four outputs represented raw class logits.
- CrossEntropyLoss was used because this is a single-label, four-class classification problem.
- Validation loss was used for early stopping and model selection. The best validation model was restored before final test evaluation.
- The neural-network results were then compared against Logistic Regression using the same embedding features.
- This experiment demonstrates that pretrained semantic representations and classifier complexity are separate design decisions. A more complex classifier does not automatically provide better generalization, especially
- for a small dataset.

##### 41. Final Architecture Learned So Far

At this point, experimentally covered:

``` text

Raw support-ticket text
        ↓
Cleaning / tokenization
        ↓
Bag of Words
        ↓
TF-IDF
        ↓
Logistic Regression

```

then:

``` text

Raw support-ticket text
        ↓
Sentence embeddings
        ↓
Logistic Regression

```

and now:

``` text

Raw support-ticket text
        ↓
Sentence embeddings
        ↓
Neural Network
        ↓
Multiclass prediction

```

##### Next notebook

10 — transformer_foundations.

That notebook will answer a different question: rather than only using the final sentence embedding produced by a pretrained transformer, we'll understand what is happening inside the transformer itself — 
tokens → token embeddings → positional information → self-attention → contextual representations. 
That bridge is especially important before we build the transformer ticket classifier in Notebook 11.